# Ejercicios del Día 4 · Medir si funciona

### Sesiones 10, 11 y 12

Estos ejercicios son para que practiques por tu cuenta lo que acabamos de ver en clase. No son
un examen: puedes equivocarte todas las veces que quieras y volver a ejecutar.

Cada ejercicio funciona igual. Hay una celda donde tú escribes algo, y debajo una celda de
comprobación que te dice si va bien y, si no, qué revisar. Ejecuta las dos.

Antes de empezar, conviene que hayas ejecutado `09_Evaluacion_y_metricas.ipynb`.

Tiempo estimado: 35 minutos.

In [ ]:
# Esta celda prepara la comprobación de los ejercicios. Ejecútala primero.

def comprobar(numero, condicion, bien, mal):
    """Revisa una respuesta y explica el resultado."""
    marca = "CORRECTO" if condicion else "REVISAR"
    print(f"[{marca}] Ejercicio {numero}")
    print(f"   {bien if condicion else mal}")


print("Listo. Ya puedes resolver los ejercicios.")

## Ejercicio 4.1 · Precisión y cobertura a mano

Antes de calcularlas con código, conviene hacerlas una vez a mano.

Un sistema recibe la pregunta "¿cuánto tarda el envío?". El gold set dice que la respuesta está
en `DOC-ENV-01`. El sistema, con k=4, recupera estos documentos:

    DOC-COS-01, DOC-ENV-01, DOC-ENV-01, DOC-DEV-01

Fíjate en que `DOC-ENV-01` aparece dos veces, porque vinieron dos fragmentos del mismo documento.
Cuenta documentos distintos, no fragmentos.

In [ ]:
# Calcula a mano y escribe los tres números.
# Recuerda: precisión = aciertos / recuperados ; cobertura = aciertos / esperados
# El rango recíproco es 1 dividido entre la posición del primer acierto.

precision = None      # por ejemplo 0.5
cobertura = None
rango_reciproco = None

In [ ]:
ok = (precision is not None and abs(precision - 1/3) < 0.01
      and cobertura == 1.0 and abs(rango_reciproco - 0.5) < 0.01)

comprobar("4.1", ok,
          "Correcto. Precisión 1/3 porque de tres documentos distintos solo uno servía; "
          "cobertura 1.0 porque el único esperado sí llegó; rango recíproco 0.5 porque "
          "apareció en segundo lugar.",
          "Revisa: son tres documentos DISTINTOS (COS, ENV, DEV), el esperado es uno solo, "
          "y el primer acierto está en la segunda posición.")

## Ejercicio 4.2 · Evaluar el sistema completo

Monta el sistema, corre las preguntas del gold set y calcula la cobertura media con k=3.

In [ ]:
import csv, re, warnings
warnings.filterwarnings("ignore")
from pathlib import Path

from langchain_community.vectorstores import LanceDB
from langchain_core.documents import Document
from langchain_ollama import OllamaEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

documentos = []
for ruta in sorted(Path("corpus/corpus_tiendasol").glob("*.md")):
    t = ruta.read_text(encoding="utf-8")
    documentos.append(Document(page_content=t,
                               metadata={"doc_id": re.search(r"doc_id:\s*(\S+)", t).group(1)}))

fragmentos = RecursiveCharacterTextSplitter(
    chunk_size=500, chunk_overlap=100).split_documents(documentos)
almacen = LanceDB.from_documents(fragmentos, OllamaEmbeddings(model="embeddinggemma:300m"),
                                 uri="/tmp/lancedb_ej4", table_name="ej", mode="overwrite")

with open(Path("corpus") / "gold_set_tiendasol.csv", encoding="utf-8") as f:
    gold = list(csv.DictReader(f))

print(f"{len(fragmentos)} fragmentos y {len(gold)} preguntas. Listo.")

In [ ]:
# ESCRIBE TU CÓDIGO AQUÍ
# Para cada pregunta del gold set que TENGA documentos esperados (las no contestables
# se saltan), recupera k=3, quédate con los doc_id distintos, y calcula
# cuántos de los esperados aparecieron dividido entre cuántos se esperaban.
# Luego promedia todas esas coberturas.

cobertura_media = None

In [ ]:
comprobar("4.2",
          cobertura_media is not None and abs(cobertura_media - 0.848) < 0.02,
          f"Correcto: cobertura media {cobertura_media:.3f}. "
          "Coincide con lo que medimos en clase.",
          "Debería dar alrededor de 0.848. Revisa que saltes las preguntas sin "
          "documentos esperados y que cuentes documentos distintos.")

## Ejercicio 4.3 · Dónde falla, por tipo de pregunta

Un promedio no dice dónde trabajar. Calcula la cobertura por separado para las preguntas de tipo
`facil` y las de tipo `sinonimo`.

In [ ]:
# ESCRIBE TU CÓDIGO AQUÍ

cobertura_facil = None
cobertura_sinonimo = None

In [ ]:
ok = (cobertura_facil is not None and cobertura_sinonimo is not None
      and cobertura_facil > cobertura_sinonimo)

comprobar("4.3", ok,
          f"Correcto: fácil {cobertura_facil:.2f} contra sinónimo {cobertura_sinonimo:.2f}. "
          "Ahí está la diferencia que un promedio global esconde.",
          "Las preguntas fáciles deberían salir mejor que las de sinónimo. "
          "Revisa el filtro por categoría.")

## Ejercicio 4.4 · Un KPI que engaña

El gerente de servicio propone medir el éxito del chatbot con un solo indicador:

> "Tasa de contención: porcentaje de conversaciones que el bot resuelve sin pasar a un humano.
> Meta: 85%."

Escribe abajo cómo podría un equipo alcanzar esa meta empeorando el servicio al cliente, y qué
segundo indicador pondrías al lado para evitarlo.

In [ ]:
mi_respuesta = """
ESCRIBE AQUÍ TU RESPUESTA
"""

print(mi_respuesta)

Compara con esto: basta con dificultar el paso a un agente humano. Si se esconde el botón de
"hablar con una persona", la contención sube y la experiencia empeora. También sirve dejar que el
bot conteste siempre algo, aunque no sepa, porque una respuesta inventada cuenta como contenida.

El indicador que hay que poner al lado es alguno que capture si el cliente quedó resuelto: la
satisfacción de quienes califican, o mejor, la proporción de clientes que vuelven a preguntar lo
mismo poco después. Una contención alta con reincidencia alta significa que el bot atrapa, no que
resuelve.

## Para cerrar

Si algún ejercicio te quedó marcado como REVISAR y no encuentras por qué, anótalo y pregúntalo
mañana al empezar. Es información útil: si a varios les pasó lo mismo, conviene revisarlo con
todo el grupo.

Y si terminaste antes de tiempo, la mejor forma de aprovechar el rato es volver al cuaderno del
día y cambiar cosas a propósito para ver qué se rompe. Aprender qué rompe un sistema enseña más
que verlo funcionar.